# Lesson 10：LLM Agents

- **讲师**：Harrison Chase（LangChain 联合创始人）
- **发布日期**：2023 年 5 月
- **课程资源**：
  - [课程视频](https://www.youtube.com/watch?v=DWUdGhRrv2c&list=PL1T8fO7ArWleyIqOy37OVXsP4hFXymdOZ&index=10)
  - [课程主页](https://fullstackdeeplearning.com/llm-bootcamp/spring-2023/chase-agents/)
  - [课程 Slides](https://docs.google.com/presentation/d/1EDmM1R0AcstfjadCUpvoa8h7NGwNqE2dwtJXqRK1618/edit)

---

## 课程大纲

1. 什么是 Agent，为什么需要它？
2. ReAct 框架：推理与行动的结合
3. 生产环境的四大挑战
4. 近期 Agent 项目巡览
5. Agent 的未来方向

## 1. 什么是 Agent，为什么需要它？

### 核心定义

> **Agent** = 将语言模型作为推理引擎，根据用户输入**自主决定**如何与外部世界交互。

与传统的 LLM Chain（固定步骤、固定工具调用顺序）不同，Agent 能够：
- 动态选择使用哪个工具、什么时候使用
- 根据工具返回结果调整后续行动
- 处理固定 Chain 无法应对的多跳（multi-hop）任务和边缘情况

### Agent 的基本工作循环

```
用户输入
    ↓
语言模型（推理引擎）
    → 选择工具 + 生成工具输入
    ↓
工具执行（搜索、代码解释器、数据库查询等）
    ↓
观察结果（工具输出反馈给 LM）
    ↓
语言模型（判断：继续使用工具？还是返回最终答案？）
    ↓
最终答案输出给用户
```

这个循环持续运行，直到触发停止条件（模型输出「最终答案」标记，或达到最大步数限制）。

### 为什么不用固定 Chain？

| 场景 | 固定 Chain | Agent |
|------|-----------|-------|
| 工具调用顺序已知 | ✅ 简单可靠 | 不必要 |
| 需要根据中间结果决策 | ❌ 无法处理 | ✅ 灵活应对 |
| 多跳问题（需要多次查询） | ❌ 难以设计 | ✅ 自主规划 |
| 边缘情况处理 | ❌ 需要枚举所有情况 | ✅ 自适应 |

## 2. ReAct 框架：推理与行动的结合

### ReAct 是什么？

**ReAct**（Reasoning + Acting）是目前最主流的 Agent 框架，来自 Yao et al. 的学术论文。核心思想是：

> 让模型在**执行动作之前先输出推理过程**，使推理和行动交替进行。

### 三步循环：Thought → Action → Observation

```
Thought（思考）：分析当前情况，规划下一步
    ↓
Action（行动）：选择工具并生成调用参数
    ↓
Observation（观察）：获取工具执行结果
    ↓
Thought（再次思考）：结合新信息调整策略
    ↓
... 循环直到输出 Final Answer
```

### ReAct 相比纯 Action 的优势

| 优势 | 说明 |
|------|------|
| **可解释性** | Thought 步骤使推理过程透明，便于调试 |
| **自我纠错** | 模型可以在 Thought 中发现错误并调整策略 |
| **多步规划** | 显式推理有助于处理需要多步骤的复杂任务 |
| **与 CoT 的关系** | ReAct 是 Chain-of-Thought 在 Agent 场景下的自然延伸 |

### 示例：多跳问答（伪代码展示）

```
用户问题："2023年世界人口是多少，比2010年增长了多少？"

Thought: 需要查询2023年和2010年的世界人口数据，然后计算差值。
Action: Search("2023年世界人口")
Observation: 2023年世界人口约为80.45亿。

Thought: 得到了2023年数据，现在查询2010年数据。
Action: Search("2010年世界人口")
Observation: 2010年世界人口约为69.09亿。

Thought: 两个数据都有了，计算差值：80.45 - 69.09 = 11.36亿。
Final Answer: 2023年世界人口约为80.45亿，比2010年增长了约11.36亿。
```

## 3. 生产环境的四大挑战

Harrison 强调：将 Agent Demo 打磨成可靠的生产系统，需要专门解决以下四个核心挑战。

---

### 挑战 1：工具控制（Tool Control）

**问题**：Agent 不知道什么时候该用工具、该用哪个工具，或者对不需要工具的问题也强行调用工具。

**解决方案**：
- **高质量的工具描述**：工具的 description 是模型选择工具的主要依据，必须清晰、准确
- **工具集检索**：当可用工具数量很多时，先用检索系统找到最相关的工具子集，再交给 Agent
- **显式「直接回答」选项**：给 Agent 一个专门的「不使用工具、直接回答」选项，避免不必要的工具调用

---

### 挑战 2：输出解析（Output Parsing）

**问题**：LM 的输出是自由文本，但程序需要结构化数据（如 JSON）来调用工具。格式不一致会导致解析失败。

**解决方案**：
- **专用输出解析器**：LangChain 提供的 Output Parser 组件，负责将 LM 输出转换为结构化格式
- **带重试的错误纠正**：解析失败时，将错误信息反馈给模型，让其重新生成符合格式的输出

> 这种「自我纠错」机制显著提升了生产环境中的鲁棒性，是实用化的关键。

---

### 挑战 3：长期记忆（Long-term Memory）

**问题**：上下文窗口有限，长时间运行的 Agent 无法记住所有历史交互。

**解决方案**：混合检索策略

```
Agent 的工作记忆 = 最近 N 步（保证连贯性）
                 + 最相关 K 步（向量检索历史中最相关的片段）
```

| 记忆类型 | 实现方式 | 特点 |
|----------|----------|------|
| **情节记忆（Episodic）** | 向量数据库存储历史交互 | 能回忆特定过去事件 |
| **语义记忆（Semantic）** | RAG + 知识库 | 访问领域知识 |
| **程序记忆（Procedural）** | 优化工具使用模式 | 学习「怎么做」的技能 |

---

### 挑战 4：评估（Evaluation）

**问题**：Agent 的评估比普通 LLM 任务更复杂——不仅最终答案要正确，中间步骤也要合理。

**解决方案**：评估两个维度

1. **最终答案评估**：最终输出是否正确、有用？
2. **轨迹评估（Trajectory Evaluation）**：中间步骤是否高效、正确？
   - 是否选择了正确的工具？
   - 是否走了不必要的弯路？
   - 工具调用的参数是否合理？

> **核心洞见**：仅评估最终答案是不够的。一个「碰巧答对」但推理过程混乱的 Agent 在新场景下会失败。轨迹评估帮助识别系统性的推理缺陷。

## 4. 近期 Agent 项目巡览

Harrison 在课程中回顾了 2023 年几个有影响力的 Agent 项目，展示了这一领域的快速发展。

---

### AutoGPT

**定位**：面向长期、开放性目标的自主 Agent（例如「帮我增加 Twitter 粉丝数」）

**关键创新**：
- 引入**持久化向量存储**作为 Agent 的长期记忆
- 首次大规模展示了「完全自主运行」的 Agent 可能性
- GitHub 发布后迅速成为最受关注的开源项目之一

**暴露的问题**：可靠性差，偶尔能完成令人印象深刻的任务，但经常陷入循环或走偏，说明从 Demo 到生产之间存在巨大工程鸿沟。

---

### BabyAGI

**定位**：将任务规划和任务执行解耦的 Agent 架构

```
任务队列（Task Queue）
    ↓
规划 LM（Planning LM）：决定下一步做什么，生成新子任务
    ↓
执行 LM（Execution LM）：执行具体任务
    ↓
结果存入记忆 → 更新任务队列 → 循环
```

分离「想什么做」和「怎么做」，让各自的 LM 专注于自己的角色，为后来的 multi-agent 架构提供了设计参考。

---

### CAMEL（多 Agent 协作）

**定位**：探索多 Agent 协作完成任务

- **指令者 Agent（Instructor）**：负责给出指令和方向
- **执行者 Agent（Assistant）**：负责具体执行任务
- 两个 Agent 通过自然语言对话协作，共同完成目标
- 展示了多个专门化 Agent 分工协作可以胜任单个 Agent 难以处理的复杂任务

---

### Generative Agents（Stanford 论文）

**定位**：模拟具有社会行为的 AI 智能体

**实验**：在虚拟小镇中模拟 25 个 Agent 的日常生活和社交互动，涌现出信息传播、自发组织活动等社交行为。

**关键技术——记忆检索权重**：

| 权重维度 | 说明 |
|----------|------|
| **时近性（Recency）** | 最近发生的事权重更高 |
| **重要性（Importance）** | 被认为重要的事件权重更高 |
| **相关性（Relevance）** | 与当前情境相关的记忆权重更高 |

还引入了**反思机制（Reflection）**：Agent 定期对经历进行高层次总结，形成抽象认知。

---

### 项目对比汇总

| 项目 | 核心贡献 | 主要局限 |
|------|----------|----------|
| **AutoGPT** | 长期自主 Agent + 向量记忆 | 可靠性低 |
| **BabyAGI** | 规划与执行分离架构 | 任务范围有限 |
| **CAMEL** | 多 Agent 协作框架 | 对话效率低 |
| **Generative Agents** | 真实感记忆与反思机制 | 计算成本极高 |

## 5. Agent 的未来方向

---

### 长时间运行的 Agent（Long-horizon Agents）

- 当前 Agent 通常处理几分钟内能完成的任务
- 未来目标：自主工作**数小时乃至数天**，处理真正复杂的长期任务
- 关键挑战：长时间运行中保持目标一致性、管理不断增长的记忆

---

### 多 Agent 架构（Multi-agent Architectures）

由一个**管理者 Agent** 协调多个**专门化子 Agent**，类比团队中的项目经理与各专业执行者：

```
用户目标
    ↓
管理者 Agent（Orchestrator）
    ├─ 搜索子 Agent
    ├─ 代码子 Agent
    └─ 数据分析子 Agent
```

---

### 可观察性与调试（Observability）

> Harrison 特别强调这一方向，后来演化为 **LangSmith** 产品。

- Agent 的推理链路必须**可追踪、可调试**
- 需要工具记录完整的 Thought-Action-Observation 轨迹
- 没有可观察性，生产环境中的 Agent 调试几乎不可能

**关键实践**：
- 每次工具调用都要记录输入输出
- 保存完整的 Agent 执行轨迹供事后分析
- 将失败案例直接转化为测试用例，驱动持续改进

---

### 其他方向

- **Agent 个性化**：学习并记住个别用户的偏好和习惯，随使用时间增长越来越贴合用户需求
- **自主研究系统**：端到端完成信息采集 → 综合分析 → 事实核查 → 生成报告
- **Human-in-the-Loop**：高风险决策时暂停并请求人类确认，随可靠性提升逐渐减少人工介入频率

---

## 总结：核心认知

| 主题 | 关键认知 |
|------|----------|
| **Agent 本质** | LLM 作为推理引擎，动态决策如何与外部世界交互 |
| **ReAct 框架** | Thought→Action→Observation 循环，显式推理提升可靠性 |
| **生产挑战** | 工具控制、输出解析、长期记忆、轨迹评估是四大核心问题 |
| **LangChain 价值** | 提供可组合的抽象层，让开发者专注于应用逻辑而非底层工程 |
| **可观察性** | 没有可追踪的推理轨迹，生产 Agent 无法可靠调试和改进 |
| **发展趋势** | 长时间运行、多 Agent 协作、个性化是三大核心方向 |